# 4 · Change Data Feed

The dashboard counts users by interest. How does it learn what changed?

In [1]:
#@title Setup — run before the session
import os, shutil, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==3.5.3', 'delta-spark==3.3.2'], check=True)
if not shutil.which('java'):
    subprocess.run('apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless', shell=True, check=True)

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (SparkSession.builder.master('local[2]')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.databricks.delta.properties.defaults.enableDeletionVectors', 'false')
    .config('spark.ui.showConsoleProgress', 'false'))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('OFF')  # errors still reach Python

PATH = '/tmp/delta_talk/cdf'
shutil.rmtree(PATH, ignore_errors=True)
print('Spark', spark.version, 'ready')

Spark 3.5.3 ready


In [2]:
#@title Helpers
from pyspark.sql import functions as F
SCHEMA = 'id INT, gender STRING, age INT, interest STRING'

## Our table, with the change feed switched on

In [3]:
profiles = spark.createDataFrame(
    [(1, 'Male', 25, 'Sports'), (2, 'Female', 30, 'News'),
     (3, 'Male', 35, 'Social'), (4, 'Female', 28, 'Fashion')], SCHEMA)
profiles.coalesce(1).write.format('delta').option('delta.enableChangeDataFeed', 'true').save(PATH)
table = DeltaTable.forPath(spark, PATH)

spark.read.format('delta').load(PATH).groupBy('interest').count().orderBy('interest').show()

+--------+-----+
|interest|count|
+--------+-----+
| Fashion|    1|
|    News|    1|
|  Social|    1|
|  Sports|    1|
+--------+-----+



## Three changes: user 1 moves to News, user 3 leaves, user 5 joins

In [4]:
table.update("id = 1", {'interest': "'News'"})
table.delete("id = 3")
spark.createDataFrame([(5, 'Male', 22, 'Social')], SCHEMA).coalesce(1).write.format('delta').mode('append').save(PATH)

## What changed since version 0?

In [5]:
changes = (spark.read.format('delta')
    .option('readChangeFeed', 'true')
    .option('startingVersion', 1)
    .load(PATH))

changes.select('_commit_version', '_change_type', 'id', 'interest').orderBy('_commit_version', '_change_type').show()

+---------------+----------------+---+--------+
|_commit_version|    _change_type| id|interest|
+---------------+----------------+---+--------+
|              1|update_postimage|  1|    News|
|              1| update_preimage|  1|  Sports|
|              2|          delete|  3|  Social|
|              3|          insert|  5|  Social|
+---------------+----------------+---+--------+



## Turn each change into +1 or −1

In [6]:
diff = changes.groupBy('interest').agg(F.sum(
    F.when(F.col('_change_type').isin('insert', 'update_postimage'), 1).otherwise(-1)).alias('change'))
diff.orderBy('interest').show()

+--------+------+
|interest|change|
+--------+------+
|    News|     1|
|  Social|     0|
|  Sports|    -1|
+--------+------+



## Old dashboard + changes = new dashboard?

In [7]:
old = spark.read.format('delta').option('versionAsOf', 0).load(PATH).groupBy('interest').count()
old.join(diff, 'interest', 'full_outer').select('interest',
    (F.coalesce('count', F.lit(0)) + F.coalesce('change', F.lit(0))).alias('count')) \
   .where('count > 0').orderBy('interest').show()

spark.read.format('delta').load(PATH).groupBy('interest').count().orderBy('interest').show()

+--------+-----+
|interest|count|
+--------+-----+
| Fashion|    1|
|    News|    2|
|  Social|    1|
+--------+-----+

+--------+-----+
|interest|count|
+--------+-----+
| Fashion|    1|
|    News|    2|
|  Social|    1|
+--------+-----+



**What if the dashboard is offline for a month?** → slides